# 🛍️ Customer Segmentation Using RFM Analysis
### E-Commerce Behavioral Analytics | UCI Online Retail Dataset

**Objective:** Segment customers based on purchasing behavior to identify 
high-value customers, at-risk churners, and growth opportunities.

**Tools:** Python · pandas · matplotlib · seaborn · plotly

**Dataset:** UCI Online Retail Dataset — 541,909 transactions from a UK-based 
online retailer (2010–2011)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✅ Libraries loaded successfully")

In [ ]:
# Load dataset
df = pd.read_excel('C:\\Project\\customer-segmentation-rfm\\data\\Online Retail.xlsx', engine='openpyxl')

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 1. Exploratory Data Analysis (EDA)

Before cleaning, let's understand what we're working with.

In [ ]:
print("=== BASIC INFO ===")
print(f"Total rows     : {df.shape[0]:,}")
print(f"Total columns  : {df.shape[1]}")
print(f"Date range     : {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"Unique products : {df['StockCode'].nunique():,}")
print(f"Countries       : {df['Country'].nunique()}")

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DATA TYPES ===")
print(df.dtypes)

## 2. Data Cleaning

Issues to address:
- Remove rows with missing `CustomerID` (can't do RFM without it)
- Remove cancelled orders (InvoiceNo starting with 'C')
- Remove rows where Quantity or UnitPrice ≤ 0
- Keep only United Kingdom for focused analysis

In [ ]:
df_clean = df.copy()

# Drop missing CustomerID
df_clean = df_clean.dropna(subset=['CustomerID'])

# Remove cancellations
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Remove negative/zero quantity and price
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Focus on UK customers
df_clean = df_clean[df_clean['Country'] == 'United Kingdom']

# Create TotalPrice column
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Ensure correct date type
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print(f"Rows before cleaning : {df.shape[0]:,}")
print(f"Rows after cleaning  : {df_clean.shape[0]:,}")
print(f"Rows removed         : {df.shape[0] - df_clean.shape[0]:,}")
print(f"\nCustomers remaining  : {df_clean['CustomerID'].nunique():,}")

## 3. RFM Calculation

- **Recency (R):** How recently did the customer make a purchase?
- **Frequency (F):** How often do they purchase?
- **Monetary (M):** How much total revenue do they generate?

In [ ]:
# Snapshot date = 1 day after last transaction
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date.date()}")

# Calculate RFM
rfm = df_clean.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

print(f"\nRFM Table Shape: {rfm.shape}")
rfm.describe()

In [7]:
# Score each metric 1-5 (5 = best)
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  q=5, labels=[1,2,3,4,5])

# Combine into RFM Score string
rfm['RFM_Score'] = (rfm['R_Score'].astype(str) + 
                    rfm['F_Score'].astype(str) + 
                    rfm['M_Score'].astype(str))

# Total RFM Score (numeric)
rfm['RFM_Total'] = (rfm['R_Score'].astype(int) + 
                    rfm['F_Score'].astype(int) + 
                    rfm['M_Score'].astype(int))

rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total
0,12346.00,326,1,77183.60,1,1,5,115,7
1,12747.00,2,11,4196.01,5,5,5,555,15
2,12748.00,1,209,33719.73,5,5,5,555,15
3,12749.00,4,5,4090.88,5,4,5,545,14
4,12820.00,3,4,942.34,5,4,4,544,13
5,12821.00,214,1,92.72,1,1,1,111,3
6,12822.00,71,2,948.88,3,2,4,324,9
7,12823.00,75,5,1759.50,2,4,4,244,10
8,12824.00,60,1,397.12,3,1,2,312,6
9,12826.00,3,7,1474.72,5,5,4,554,14
